# Exudate Segmentation - SVM / Logistic Regression vs U-Net

This notebook is only the **Google Colab runner**. The implementation lives in `src/`, so Colab only installs dependencies, clones the repository, sets parameters, and calls `run_experiment(...)`.

**Compared methods**
1. Traditional machine learning: preprocessing -> candidate analysis -> handcrafted features -> SVM and Logistic Regression
2. Deep learning: U-Net with an ImageNet-pretrained encoder


## 1. Environment


In [ ]:
import subprocess, sys, torch

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None - please switch to a GPU runtime")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "segmentation-models-pytorch==0.3.4",
    "opencv-python-headless",
    "scikit-image",
    "scikit-learn",
    "scipy",
], check=True)
print("Dependencies installed")


## 2. Get Data and Modules

By default this clones the GitHub repository that contains both the data and the decoupled `src/` modules. If you forked or renamed the repository, only change `REPO_URL`.


In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/LeafTraces/ML-Proj.git"   # Change this if needed
PROJECT_DIR = "ML-Proj"
DATA_SUBDIR = "topic/眼底图像的渗出液分割/EX数据"

if not os.path.isdir(PROJECT_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, PROJECT_DIR], check=True)

for module_path in [os.getcwd(), PROJECT_DIR]:
    if module_path not in sys.path:
        sys.path.insert(0, module_path)

DATA_ROOT = os.path.join(PROJECT_DIR, DATA_SUBDIR)
assert os.path.isdir(os.path.join(DATA_ROOT, "images")), "Data not found; check REPO_URL and DATA_SUBDIR"
assert os.path.isdir("src") or os.path.isdir(os.path.join(PROJECT_DIR, "src")), "src modules not found; make sure the repository contains the decoupled code"
print("Data root:", DATA_ROOT)


## 3. Configuration


In [ ]:
SEED = 42
WORK_W, WORK_H = 672, 448
IMG_SIZE = (WORK_W, WORK_H)
VAL_RATIO = 0.18

UNET_ENCODER = "resnet34"
EPOCHS = 60
BATCH = 8
LR = 1e-3
OUT = "outputs"


## 4. Run Experiment

This runs data loading, preprocessing, SVM/LR training, U-Net training, shared evaluation, figure generation, and result export.


In [ ]:
from src.experiment import run_experiment

df, results = run_experiment(
    data_root=DATA_ROOT,
    seed=SEED,
    work_size=IMG_SIZE,
    val_ratio=VAL_RATIO,
    unet_encoder=UNET_ENCODER,
    epochs=EPOCHS,
    batch=BATCH,
    lr=LR,
    out_dir=OUT,
)


## 5. Download Outputs

The run writes `outputs/results_table.csv`, `outputs/results.json`, and figures under `outputs/figures/`.


In [ ]:
import shutil

shutil.make_archive("exudate_outputs", "zip", OUT)
print("Created exudate_outputs.zip with figures, results.json, and results_table.csv")
try:
    from google.colab import files
    files.download("exudate_outputs.zip")
except Exception:
    print("If auto-download is unavailable, download exudate_outputs.zip from the Colab file panel")


---
### Output Files

- `outputs/results_table.csv`: shared metric table for SVM, Logistic Regression, and U-Net.
- `outputs/results.json`: raw metrics, confusion matrices, and linear-model feature coefficients.
- `outputs/figures/`: preprocessing, linear coefficients, U-Net training curve, ROC/PR, confusion matrices, and qualitative comparisons.
